# Runtime and  Memory (Empirical)  

Measures actual time and memory for each step.
In Addition to `runtime_analysis_models.ipynb` and `runtime_analysis_features.ipynb`, there you can find the O-Notation.

Time:with  `time.perf_counter()`
Memory: with `tracemalloc` (built-in)

Each model is timed with single fit on the full training set (no GridSearch overhead) -> Fairer Compearision (because of the folds)

In [1]:
import os, sys, time, tracemalloc
from pathlib import Path

def _find_root():
    _cur = Path(".").resolve()
    for p in [_cur, *_cur.parents]:
        if (p / "pyproject.toml").exists():
            return str(p)
    raise FileNotFoundError("pyproject.toml not found")

PROJECT_ROOT = _find_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from analysis_and_inference.models._common import load_split, precompute_features
from analysis_and_inference.models.lasso_log_reg.core_logistic_regression_lasso import LassoLogisticRegression
from analysis_and_inference.features.build_features import DenseFeatureTransformer

print("Done")

Done


In [2]:
#Load x features (Precomputed/ reads from cache)
X_train_raw, X_test_raw, y_train, y_test = load_split()
X_train, X_test = precompute_features(X_train_raw, X_test_raw)

print(f"Train: {X_train.shape}  ,  Test: {X_test.shape}")

[skip] analysis_and_inference/models/split_and_features/features.pkl already exists — loading saved features
Train: (127656, 32)  ,  Test: (31915, 32)


In [3]:
def measure(fn):
    """Run fn(), return (result, elapsed_seconds, peak_mb)."""
    tracemalloc.start()
    t0 = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak / 1024 ** 2  #peak in MB


## 1. Feature Engineering

`DenseFeatureTransformer` is called on the raw training text so the actual computation runs (not the cached load).

In [4]:
SAMPLE = 2000
sample = X_train_raw.iloc[:SAMPLE]
n_full = len(X_train_raw)

transformer = DenseFeatureTransformer()
_, feat_time_sample, feat_mem = measure(lambda: transformer.transform(sample))
feat_time_full = feat_time_sample * (n_full / SAMPLE)

print(f"DenseFeatureTransformer.transform()")
print(f"  sample n = {SAMPLE:,}  ->  timed  : {feat_time_sample:.2f} s")
print(f"  full   n = {n_full:,}  ->  est.   : {feat_time_full:.1f} s  ({feat_time_full/60:.1f} min)")
print(f"  peak memory (sample): {feat_mem:.1f} MB")

Generating dense features...


DenseFeatureTransformer.transform()
  sample n = 2,000  ->  timed  : 23.32 s
  full   n = 127,656  ->  est.   : 1488.4 s  (24.8 min)
  peak memory (sample): 4.6 MB


## 2. Model (Training)

GridSearch is excluded, because it depends on grid size (12 fits for Ridge/SVM, 36 for RF), not on the algorithm itself. Single fits isolate the algorithmic training cost.

In [5]:
models = {
    "Baseline": DummyClassifier(strategy="stratified", random_state=42),
    "Lasso Log Reg": LassoLogisticRegression(alpha=0.01, learning_rate=0.1, max_iter=1500),
    "Ridge Log Reg": LogisticRegression(solver="lbfgs", class_weight="balanced",
                                         max_iter=1000, C=1.0, random_state=42),
    "LinearSVM": LinearSVC(class_weight="balanced", max_iter=2000, C=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10,
                                             class_weight="balanced",
                                             n_jobs=-1, random_state=42),
}

train_results = {}

for name, model in models.items():
    fitted, t, mem = measure(lambda m=model: m.fit(X_train, y_train))
    models[name] = fitted   # keep fitted model for inference step
    train_results[name] = {"time_s": round(t, 3), "peak_mb": round(mem, 1)}
    print(f"{name:<20}  {t:7.3f} s   peak {mem:.1f} MB")

Baseline                0.017 s   peak 5.0 MB


Lasso Log Reg          16.856 s   peak 6.8 MB


Ridge Log Reg           2.596 s   peak 37.1 MB


LinearSVM               5.093 s   peak 37.1 MB


Random Forest           8.546 s   peak 82.3 MB



## 3. Model (Inference)

Each fitted model runs `predict()` on `X_test`. Averaged over 3 runs reducing noise


In [6]:
infer_results = {}
RUNS = 3

for name, model in models.items():
    times = []
    for _ in range(RUNS):
        _, t, _ = measure(lambda m=model: m.predict(X_test))
        times.append(t)
    avg_t = round(sum(times) / RUNS, 4)
    infer_results[name] = {"time_s": avg_t}
    print(f"{name:<20}  {avg_t:.4f} s  (avg over {RUNS} runs)")

Baseline              0.0118 s  (avg over 3 runs)
Lasso Log Reg         0.0015 s  (avg over 3 runs)
Ridge Log Reg         0.0038 s  (avg over 3 runs)
LinearSVM             0.0037 s  (avg over 3 runs)


Random Forest         0.1704 s  (avg over 3 runs)


-------

## 4. Summary

In [7]:
o_notation = {
    "Baseline":       ("O(n)",             "O(n_test)",              "O(1)"),
    "Lasso Log Reg":  ("O(T·n·p)",         "O(n_test·p)",            "O(p)"),
    "Ridge Log Reg":  ("O(T·n·p)",         "O(n_test·p)",            "O(p)"),
    "LinearSVM":      ("O(T·n·p)",         "O(n_test·p)",            "O(p)"),
    "Random Forest":  ("O(T·n·log n·√p)",  "O(n_test·T·d)",          "O(T·2^d)"),
}

rows = []
for name in models:
    o_train, o_infer, o_space = o_notation[name]
    train_s = train_results[name]["time_s"]
    rows.append({
        "Model":                       name,
        "Train time (s)":              train_s,
        "Peak mem (MB)":               train_results[name]["peak_mb"],
        "Infer time (s)":              infer_results[name]["time_s"],
        "Total incl. feat. eng. (s)":  round(feat_time_full + train_s, 1),
        "O train":                     o_train,
        "O infer":                     o_infer,
        "O space (model)":             o_space,
    })

df = pd.DataFrame(rows).set_index("Model")
df

,Train time (s),Peak mem (MB),Infer time (s),Total incl. feat. eng. (s),O train,O infer,O space (model)
Model,,,,,,,
Baseline,0.017,5.0,0.0118,1488.4,O(n),O(n_test),O(1)
Lasso Log Reg,16.856,6.8,0.0015,1505.2,O(T·n·p),O(n_test·p),O(p)
Ridge Log Reg,2.596,37.1,0.0038,1491.0,O(T·n·p),O(n_test·p),O(p)
LinearSVM,5.093,37.1,0.0037,1493.5,O(T·n·p),O(n_test·p),O(p)
Random Forest,8.546,82.3,0.1704,1496.9,O(T·n·log n·√p),O(n_test·T·d),O(T·2^d)
